# Joint lesion segmentation and classification (public data)

Compare a segmentation-only CNN with the same architecture trained jointly on segmentation and an image-level lesion-size category. The two outputs share an encoder. The auxiliary label is whether foreground area exceeds the median training-mask area. Models select checkpoints by validation Dice. An auxiliary AUROC is reported for the joint arm. Use 520 Kvasir-SEG polyp images and masks at 40×40. Train and evaluate on disjoint images within each repeat. The auxiliary label measures polyp area, with no malignancy prediction. The task adapts the joint training idea in Mudduluru's work.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'joint-lesion-learning-real-data-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2', 'joblib==1.6.0', 'torch==2.14.1', 'Pillow==12.3.0']],check=True)

In [ ]:
(root / 'prepare.py').write_text('import hashlib\nimport os\nimport zipfile\nfrom pathlib import Path\nfrom urllib.request import urlretrieve\n\nroot = (\n    Path(os.environ.get("PORTFOLIO_DATA", Path(__file__).resolve().parent / "data"))\n    / "kvasir-seg"\n)\nroot.mkdir(parents=True, exist_ok=True)\ntarget = root / "kvasir-seg.zip"\nif not target.exists():\n    temp = target.with_suffix(".part")\n    urlretrieve("https://datasets.simula.no/downloads/kvasir-seg.zip", temp)\n    temp.replace(target)\nassert (\n    hashlib.sha256(target.read_bytes()).hexdigest()\n    == "03b30e21d584e04facf49397a2576738fd626815771afbbf788f74a7153478f7"\n)\nwith zipfile.ZipFile(target) as archive:\n    archive.extractall(root / "extracted")\n',encoding='utf8')
subprocess.run([sys.executable,'prepare.py'],cwd=root,check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom sklearn.metrics import (\n    average_precision_score,\n    roc_auc_score,\n)\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", ROOT / "data"))\nSEEDS = [13, 37, 61]\n\n\ndef save(obj, name):\n    folder = ROOT / "checkpoints"\n    folder.mkdir(exist_ok=True)\n    joblib.dump(obj, folder / (name + ".joblib"), compress=3)\n\n\ndef scores(y, p):\n    return {\n        "auroc": float(roc_auc_score(y, p)),\n        "average_precision": float(average_precision_score(y, p)),\n    }\n\n\nfrom PIL import Image\n\n\ndef load_images(seed):\n    base = DATA / "kvasir-seg/extracted/Kvasir-SEG"\n    files = sorted((base / "images").glob("*.jpg"))\n    assert len(files) == 1000\n    ids = np.random.default_rng(902).choice(len(files), 520, replace=False)\n    x = []\n    y = []\n    for i in ids:\n        path = files[i]\n        with Image.open(path) as im:\n            x.append(np.asarray(im.convert("RGB").resize((40, 40))) / 255.0)\n        with Image.open(base / "masks" / path.name) as im:\n            y.append(\n                np.asarray(im.convert("L").resize((40, 40), Image.Resampling.NEAREST))\n                > 127\n            )\n    return np.asarray(x), np.asarray(y)\n\n\ndef dataset(seed):\n    x, y = load_images(seed)\n    tr, hold = train_test_split(np.arange(len(y)), test_size=0.36, random_state=seed)\n    va, te = np.array_split(hold, 2)\n    return x, y, tr, va, te\n\n\nimport torch\nfrom sklearn.model_selection import train_test_split\nfrom torch import nn\nfrom torch.nn import functional as F\n\ntorch.set_num_threads(2)\n\n\nclass Net(nn.Module):\n    def __init__(self, channels):\n        super().__init__()\n        self.stem = nn.Sequential(nn.Conv2d(channels, 12, 3, padding=1), nn.ReLU())\n        self.context = nn.Sequential(\n            nn.MaxPool2d(2),\n            nn.Conv2d(12, 20, 3, padding=1),\n            nn.ReLU(),\n            nn.Conv2d(20, 20, 3, padding=1),\n            nn.ReLU(),\n        )\n        self.mask = nn.Sequential(\n            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(), nn.Conv2d(16, 1, 1)\n        )\n        self.classifier = nn.Linear(20, 1)\n\n    def forward(self, x):\n        low = self.stem(x)\n        z = self.context(low)\n        up = F.interpolate(z, size=low.shape[-2:], mode="bilinear", align_corners=False)\n        return self.mask(torch.cat((low, up), 1)), self.classifier(\n            z.mean((2, 3))\n        ).flatten()\n\n\ndef accuracy(logits, y):\n    mask = logits >= 0\n    overlap = (mask * y).sum((1, 2, 3))\n    return (\n        ((2 * overlap + 1) / (mask.sum((1, 2, 3)) + y.sum((1, 2, 3)) + 1)).mean().item()\n    )\n\n\ndef run(seed):\n    images, masks, tr, va, te = dataset(seed)\n    images = torch.tensor(images, dtype=torch.float32).permute(0, 3, 1, 2)\n    masks = torch.tensor(masks, dtype=torch.float32)[:, None]\n    areas = masks.mean((1, 2, 3))\n    threshold = areas[tr].median()\n    category = (areas > threshold).float()\n    schemes = [("single_task", images, False), ("joint", images, True)]\n    out = {}\n    for name, features, aux in schemes:\n        torch.manual_seed(seed)\n        model = Net(features.shape[1])\n        opt = torch.optim.AdamW(model.parameters(), lr=0.003)\n        best = -1\n        for _ in range(10):\n            model.train()\n            for batch in torch.randperm(len(tr)).split(16):\n                ids = tr[batch.numpy()]\n                p, c = model(features[ids])\n                y = masks[ids]\n                soft = p.sigmoid()\n                overlap = (2 * (soft * y).sum() + 1) / (soft.sum() + y.sum() + 1)\n                weight = (1 - y.mean()) / y.mean().clamp(min=0.02)\n                loss = (\n                    F.binary_cross_entropy_with_logits(\n                        p, y, pos_weight=weight.clamp(max=12)\n                    )\n                    + 1\n                    - overlap\n                )\n                if aux:\n                    loss += 0.45 * F.binary_cross_entropy_with_logits(c, category[ids])\n                opt.zero_grad()\n                loss.backward()\n                opt.step()\n            model.eval()\n            with torch.no_grad():\n                val = accuracy(model(features[va])[0], masks[va])\n            if val > best:\n                best = val\n                state = {k: v.detach().clone() for k, v in model.state_dict().items()}\n        model.load_state_dict(state)\n        folder = ROOT / "checkpoints"\n        folder.mkdir(exist_ok=True)\n        torch.save(state, folder / f"{name}-{seed}.pt")\n        with torch.no_grad():\n            pred, logits = model(features[te])\n            result = {"dice": accuracy(pred, masks[te]), "test_images": len(te)}\n            if aux:\n                result["size_auroc"] = roc_auc_score(category[te], logits)\n        out[name] = result\n    return out\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in SEEDS}\n    first = next(iter(runs.values()))\n    means = {\n        k: {m: float(np.mean([r[k][m] for r in runs.values()])) for m in v}\n        for k, v in first.items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps({"seeds": SEEDS, "runs": runs, "means": means}, indent=2)\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())